# Katachi con imágenes FITS de DESI

Proyecto *Inferencia de propiedades físicas de galaxias a partir de imágenes multibanda* (ITESM · IA-UNAM).

Katachi (Alfonzo et al. 2024) es una red que estima la masa estelar, la tasa de formación estelar y el índice D4000 de una galaxia a partir de su imagen. Se entrenó con imágenes en color de SDSS. En este cuaderno la entrenamos de nuevo, con la misma receta, usando los cubos FITS de DESI DR10, y comparamos el resultado con el de Katachi sobre las mismas galaxias.

El diseño completo, la auditoría del código original y las decisiones del equipo están en `Documentación/Plan_Katachi_FITS.md`. El código vive en `src/katachi_fits/` y tiene pruebas automáticas en `tests/katachi_fits/`.

**Resultados** (se completa al terminar, solo con números que aparecen más abajo):

| | M\* [dex] | SFR [dex] | D4000 |
|---|---|---|---|
| Katachi publicado (B0), RMSE en Test | | | |
| Esta corrida (E1), RMSE en Test | | | |
| Diferencia E1 − B0 [IC 95 %] | | | |
| Veredicto | | | |

### Cómo usar este cuaderno

1. Abrirlo en Colab con GPU (A100 o L4) y memoria alta.
2. Revisar la celda de parámetros de la sección 2. Lo demás no debería necesitar cambios.
3. Ejecutar todo en orden. La primera vez se construye una caché de imágenes y se entrena la red, lo que lleva varias horas. Si Colab se desconecta, basta con volver a ejecutar todo: cada paso retoma donde se quedó.
4. Al terminar, cambiar `ABRIR_TEST = True` y ejecutar de nuevo para obtener los resultados en Test. Esto se hace una sola vez.

El cuaderno lee el catálogo y las imágenes, pero nunca los modifica ni los borra. Todo lo que escribe queda en `resultados_katachi_fits/`.

## 1. Qué queremos saber

Katachi encadena tres redes ResNet50. La primera estima la masa estelar a partir de la imagen. La segunda recibe la imagen y la masa estimada, y estima la tasa de formación estelar (SFR). La tercera recibe la imagen, la masa y la SFR, y estima D4000, un índice espectral que indica la edad de las estrellas. A partir de esas tres cantidades, Katachi reconstruye cuándo formó cada galaxia la mitad de sus estrellas (t50) y su historia de formación estelar, y usa mapas SHAP para ver en qué partes de la imagen se apoya la red.

Las imágenes de Katachi son PNG de SDSS: una representación en color pensada para mostrarse en pantalla. Los cubos FITS de DESI son más profundos y tienen mejor resolución. Además, cada píxel guarda el flujo medido en unidades físicas, en lugar de un color de 0 a 255.

La pregunta es: **con las mismas galaxias de prueba y las mismas etiquetas, ¿la cadena de Katachi entrenada con FITS de DESI estima mejor, peor o igual que la publicada?**

Para no elegir la métrica después de ver los resultados, la regla de decisión se fija aquí:

- La métrica principal es el RMSE de log M\* en Test. Las secundarias son el RMSE de log SFR y de D4000, y el sesgo y la dispersión robusta (σ_NMAD) de las tres.
- Diremos que E1 es **mejor** o **peor** que Katachi si el intervalo de confianza del 95 % de la diferencia de RMSE no incluye el cero.
- Diremos que es **equivalente** si el intervalo del 90 % queda dentro de ±0.02 dex (masa y SFR) o ±0.01 (D4000). Ese margen es del orden de la variación que Katachi midió al añadir ruido a las imágenes.
- En cualquier otro caso, el resultado es **no concluyente**.

Seguimos el artículo también en cómo se entrena: las capas preentrenadas de la ResNet50 quedan **congeladas** y solo se entrenan las finales (sección 7.1). Los pesos que publicó Katachi no se entrenaron así, y esa diferencia se discute más abajo.

Hay dos límites que conviene tener presentes desde el principio. Primero, esta es una sola corrida (una semilla), y entrenar con otra semilla cambia el error en alrededor de 0.01–0.02 dex. Segundo, no reentrenamos Katachi con SDSS: usamos sus resultados publicados. Por eso la diferencia que midamos combina el cambio de imágenes con las pequeñas diferencias de procedimiento que se listan en la sección 2.3.

## 2. Parámetros y supuestos

Todos los valores del experimento están en un solo lugar, el bloque `CONSTANTES` de `src/katachi_fits/config.py`. Así, cambiar un valor ahí lo cambia en todo el código. La celda de abajo muestra los que normalmente se ajustan; el resto conviene no tocarlo, porque reproduce la receta de Katachi.

### 2.1 Parámetros

| Parámetro | Valor | Qué controla | De dónde viene |
|---|---|---|---|
| `VARIANTE` | `desi` | Qué se le hace a la imagen antes de recortarla. `desi` no le hace nada (experimento E1); `desi_prep` aplicará el preprocesamiento del equipo (E2). | Diseño del experimento |
| `RECORTE_PX` | 300 | Lado del recorte central: 300 px × 0.262″ = 78.6″ | Decisión del equipo |
| `SALIDA_PX` | 300 | Tamaño de entrada a la red; igual al recorte, así que no se remuestrea | Decisión del equipo |
| `BETA` | 3 | Estiramiento `asinh(flujo / (β·σ_cielo))` | Fijado antes de entrenar (Lupton et al. 1999) |
| `SEED` | 42 | Semilla aleatoria de la corrida | — |
| `CONGELAR` | `True` | Congelar las capas preentrenadas y entrenar solo las finales. `False` da la corrida de control. | Artículo §3.1.3 |
| Capas entrenables | desde el tensor 144 | Las últimas 5 convolucionales y la lineal (7.9 M de 23.5 M parámetros) | Umbral del código de los autores |
| Orden de canales | z, r, g | Correspondencia con los filtros congelados de ImageNet (R, G, B) | Igual que Katachi: R = i, G = r, B = g |
| Filtro de calidad | QCFLAG = 0 | Solo galaxias cuyo ajuste espectral pasó todos los controles | Decisión del equipo |
| Particiones | `split_proyecto` | Train 7,123 · Validation 1,262 · Test 918 (con QCFLAG = 0) | Catálogo del equipo |
| Receta de entrenamiento | la del artículo | Lote 32; Adam 1e-3 en las tres redes; tasa de aprendizaje y parada independientes por red, según su pérdida de entrenamiento; aumentación | Artículo §3.1.2 |
| `ABRIR_TEST` | `False` | Si se evalúa en Test. Se activa una sola vez, al final. | Regla del §1 |

### 2.2 Supuestos

1. **Las etiquetas verdaderas son las de Pipe3D**, tal como están en el catálogo del equipo. Tienen sus propios errores; la sección 8.4 los compara con el error de la red.
2. **Los cubos están en nanomaggies, con el cielo ya restado.** El encabezado no declara la unidad. Lo comprobamos con una galaxia de muestra: su flujo da una magnitud compatible con la de SDSS.
3. **La galaxia está en el centro del cubo.** Las coordenadas pedidas al servidor de DESI son las del listado oficial de Katachi, y cada archivo se verifica para que su centro de referencia (CRPIX) esté en el centro.
4. **Misma transformación para todas las galaxias.** Si se normalizara cada imagen por separado, se perdería el brillo absoluto, que es la pista principal para estimar la masa.
5. **Validation no toma decisiones.** El artículo divide 90/10 sin conjunto de validación, así que la tasa de aprendizaje y la parada dependen solo de la pérdida de entrenamiento de cada red. Validation se usa únicamente para vigilar el sobreajuste.
6. **Se usan las predicciones publicadas de Katachi como línea base (B0)**, evaluadas en las mismas 918 galaxias de Test. Comprobamos que se reproducen exactamente con sus pesos y sus imágenes.
7. **Cuando el artículo es ambiguo, seguimos el código de los autores; si ambos callan, la práctica documentada de transferencia de aprendizaje.** Así se resolvió cada detalle del congelado (sección 7.1).

### 2.3 En qué nos diferenciamos de Katachi

| | Katachi | Este cuaderno |
|---|---|---|
| Imágenes | PNG de SDSS (g, r, i), 50″, 256 px | FITS de DESI (g, r, z), 78.6″, 300 px, píxeles nativos |
| Muestra | Sin filtro de calidad | QCFLAG = 0 |
| Galaxias de entrenamiento | 8,898 | 7,123, más 1,262 de Validation |
| Ajuste de la tasa de aprendizaje y parada | Su código: un planificador en la red de masa, guiado por la pérdida de Test | Como el artículo: uno por red, con su pérdida de entrenamiento |
| Tasa de aprendizaje | Código: 1e-3, 1e-3, 1e-4 | Artículo: 1e-3 en las tres |
| Capas congeladas | El artículo dice que sí; sus pesos publicados muestran que no | Sí, como dice el artículo (sección 7.1) |
| Primera capa de SFR y D4000 | Aleatoria, de 4 y 5 canales | Parte de imagen de ImageNet (congelada) y canales de masa y SFR en cero (entrenables) |
| Lo demás (arquitectura, optimizador, aumentación, parada, BatchNorm en dos mitades) | — | Igual |

In [ ]:
# ======================  PARÁMETROS DEL EXPERIMENTO  ======================
import os
env = os.environ.get          # las variables KATACHI_* solo se usan en la prueba local reducida

VARIANTE    = env("KATACHI_VARIANTE", "desi")   # "desi" (E1) o "desi_prep" (E2)
SEED        = 42
CONGELAR    = True          # True: congelado del artículo · False: corrida de control, todo entrenable
ENTRENAR    = True
ABRIR_TEST  = False         # cambiar a True una sola vez, al final
# Recorte, salida y β se toman de config.py (CROP_PX, OUTPUT_PX, ASINH_BETA); ver la siguiente celda.
# ===========================================================================

# Prueba local reducida (en Colab se dejan los valores por defecto)
LIMITE     = int(env("KATACHI_LIMIT", "0"))      # 0 = todas las galaxias
ABRIR_TEST = ABRIR_TEST or env("KATACHI_SMOKE_OPEN_TEST") == "1"

## 3. Preparación del entorno

Primero se descarga el código del repositorio en un commit concreto, para que cada resultado quede ligado a una versión exacta del código. Después se instalan las dependencias y se monta Google Drive.

In [ ]:
import sys, json, time, subprocess, datetime, platform
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/erteck/InferenciaPropiedadesFisicas.git"
REPO_REF = env("KATACHI_REPO_REF", "main")
REPO_DIR = Path(env("KATACHI_REPO_DIR", "/content/InferenciaPropiedadesFisicas"))

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hickle", "shap", "dense_basis",
                    "setuptools<81", "corner", "statsmodels", "seaborn"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "FETCH_HEAD"], check=True)

sys.path.insert(0, str(REPO_DIR))
COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Python", platform.python_version(), "· commit del código:", COMMIT[:12])

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.katachi_fits import config as K
from src.katachi_fits import catalog as C, evaluate as EV, robustness as RB, reporte as R
from src.katachi_fits.baseline import images_tensor, load_images_cat
from src.katachi_fits.cache import build_cache, cache_dir, load_cache, sky_sigma
from src.katachi_fits.chain import ChainTarget, katachi_chain, katachi_solo
from src.katachi_fits.config import KATACHI_FILES, TARGETS, Experiment, FitsConfig, Paths, TrainConfig
from src.katachi_fits.io_guard import WriteGuard, sha256
from src.katachi_fits.sfh import load_t50, predict_t50
from src.katachi_fits.train import GPUData, Trainer, freezing_audit
from src.katachi_fits.transforms import Normalizer, band_permutation, fit_scale, network_input, preprocess, resample, stretch
from src.katachi_fits.variants import DESI, get as get_variante

RECORTE_PX, SALIDA_PX, BETA = K.CROP_PX, K.OUTPUT_PX, K.ASINH_BETA
MAX_EPOCHS = int(env("KATACHI_MAX_EPOCHS", K.MAX_EPOCHS))
N_BOOT     = int(env("KATACHI_N_BOOT", K.N_BOOTSTRAP))
N_RUIDO    = int(env("KATACHI_N_NOISE", K.NOISE_ITERATIONS))
N_SHAP     = int(env("KATACHI_N_SHAP", K.N_SHAP_GALAXIES))
N_SHAP_MUESTRAS = int(env("KATACHI_SHAP_NSAMPLES", K.SHAP_NSAMPLES))
HILOS      = int(env("KATACHI_THREADS", "24"))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
print("torch", torch.__version__, "·", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

### 3.1 Dónde están los datos y dónde se escribe

El cuaderno necesita tres entradas, todas de solo lectura:

- `plan.csv`: la lista de galaxias que bajó el descargador, con la ruta de su FITS.
- La carpeta con los 9,661 FITS.
- El catálogo maestro del equipo, con etiquetas, calidad y particiones.

La única salida es la carpeta `resultados_katachi_fits/`. Un guardia en el código (`WriteGuard`) se niega a escribir en cualquier otra parte, y no hay ninguna operación de borrado.

In [ ]:
BASE       = Path(env("KATACHI_BASE", str(K.DRIVE_BASE)))
DESCARGA   = BASE / K.DOWNLOAD_DIRNAME
PLAN_CSV   = Path(env("KATACHI_PLAN", str(DESCARGA / "plan.csv")))
FITS_DIR   = Path(env("KATACHI_FITS_DIR", str(DESCARGA / "fits")))
RESULTADOS = Path(env("KATACHI_RESULTS", str(BASE / K.RESULTS_DIRNAME)))

def buscar(nombres):
    # Busca un archivo por nombre bajo BASE (solo lectura), fuera de la carpeta de resultados.
    hallados = [p for n in nombres for p in BASE.glob(f"**/{n}") if K.RESULTS_DIRNAME not in str(p)]
    assert hallados, f"No se encontró {nombres} bajo {BASE}"
    return hallados[0]

if not PLAN_CSV.exists():
    PLAN_CSV = buscar(["plan.csv"])
CATALOGO = Path(env("KATACHI_TEAM_CATALOG") or buscar(K.TEAM_CATALOG_NAMES))

paths = Paths(base=BASE, plan_csv=PLAN_CSV, inventario_csv=PLAN_CSV.parent / "inventario.csv",
              fits_dir=FITS_DIR, results=RESULTADOS, team_catalog=CATALOGO)
guard = WriteGuard(paths.results, paths.read_only)
guard.mkdir(paths.results)
print("Lectura:  ", PLAN_CSV.name, "·", CATALOGO.name, "·", FITS_DIR)
print("Escritura:", paths.results)

### 3.2 Integridad de las entradas

Antes de usar los datos comprobamos tres cosas. Primero, que el catálogo es la versión acordada: su huella SHA-256 debe coincidir con la que registró el descargador. Segundo, que `plan.csv` y el catálogo no cambian mientras corre el cuaderno; la huella se vuelve a calcular al final. Tercero, que 50 FITS elegidos al azar son idénticos, byte por byte, a los que se descargaron.

In [ ]:
SHA_PLAN, SHA_CAT = sha256(PLAN_CSV), sha256(CATALOGO)
cfg_descarga = PLAN_CSV.parent / "configuracion.json"
sha_descargador = json.loads(cfg_descarga.read_text())["catalogo_sha256"] if cfg_descarga.exists() else None
print("¿Catálogo = versión acordada?", SHA_CAT == K.TEAM_CATALOG_SHA256,
      "· ¿= catálogo del descargador?", SHA_CAT == sha_descargador)
if LIMITE == 0:
    assert SHA_CAT == K.TEAM_CATALOG_SHA256, "El catálogo no es la versión acordada"

inventario = pd.read_csv(paths.inventario_csv, dtype={"mangaid": str})
muestra = inventario.sample(min(50, len(inventario)), random_state=0)
distintos = [r.mangaid for r in muestra.itertuples() if sha256(FITS_DIR / Path(r.ruta).name) != r.sha256]
print(f"FITS idénticos a los descargados: {len(muestra) - len(distintos)} de {len(muestra)}")
assert not distintos, f"FITS modificados: {distintos[:5]}"

### 3.3 Archivos de Katachi y de Pipe3D

Se descargan una sola vez y quedan en `resultados_katachi_fits/externos/`:

- `scalars.cat`: el catálogo de Katachi, con sus predicciones publicadas.
- `images.cat`: sus imágenes SDSS, que usamos para comparar visualmente y para calcular sus mapas SHAP.
- Sus pesos entrenados.
- El catálogo público de Pipe3D, que sirve para verificar el nuestro.

In [ ]:
EXT = guard.mkdir(paths.external)
for nombre, url in KATACHI_FILES.items():
    p = guard.download(url, EXT / nombre)
    print(f"  {nombre:45s} {p.stat().st_size / 1e6:8.1f} MB")

## 4. La muestra

La fuente de verdad es el catálogo maestro del equipo (versión `20260925T190214_499654Z`). Para cada galaxia trae sus tres etiquetas con sus errores, la bandera de calidad de Pipe3D y la partición del proyecto. Las rutas de los FITS salen de `plan.csv`, y la asociación se hace siempre por `mangaid`, nunca por el orden de las filas.

Antes de usarlo lo cruzamos con dos fuentes independientes. Las etiquetas y la partición de Katachi deben coincidir con `scalars.cat`. La bandera de calidad, la observación elegida y los errores deben coincidir con el catálogo público de Pipe3D. Si algún valor difiere, el cuaderno se detiene.

In [ ]:
plan    = C.load_plan(PLAN_CSV, FITS_DIR)
scalars = C.load_scalars(EXT / "scalars.cat")
equipo  = C.load_team_catalog(CATALOGO)
vac     = C.load_pipe3d_qc(EXT / "SDSS17Pipe3D_v3_1_1.fits", scalars)

diferencias = C.cross_check(equipo, scalars, vac)
print("Diferencias con Katachi y Pipe3D (todas deben ser 0):")
print(pd.Series(diferencias).to_string())
assert all(v == 0 for v in diferencias.values())

cohorte, reporte = C.build_cohort_team(plan, equipo, scalars)
print("\nGalaxias por partición:", reporte["por_particion"])
if LIMITE == 0:
    assert reporte["por_particion"] == {"Train": 7390, "Validation": 1302, "Test": 969}

### 4.1 Filtro de calidad

`QCFLAG` es la bandera de calidad que Pipe3D asigna al ajuste espectral del que salen las etiquetas. Un 0 significa que el ajuste pasó todos los controles. Los demás códigos avisan de algo que puede volver dudosa la etiqueta: una estrella delante de la galaxia, una fusión, un AGN intenso, o una masa o un redshift que no coinciden con otro catálogo. El equipo acordó usar solo QCFLAG = 0. Las galaxias excluidas no se pierden: quedan registradas con su motivo en `exclusiones_qc.csv`.

Katachi no aplicaba este filtro. Para que la comparación siga siendo justa, sus predicciones se evalúan en las mismas 918 galaxias de Test.

In [ ]:
if LIMITE:   # prueba local: una muestra pequeña de cada partición
    cohorte = (cohorte.groupby("particion", group_keys=False)
                      .apply(lambda d: d.head(max(LIMITE // 4, 8) if d.name != "Train" else LIMITE))
                      .reset_index(drop=True))
cohorte, EXCLUIDAS = C.apply_qc(cohorte, K.QC_GROUPS_INCLUDED)
print("Galaxias excluidas por calidad:", len(EXCLUIDAS))
display(EXCLUIDAS.groupby(["motivo", "particion"]).size().unstack(fill_value=0))
print("Muestra final:", cohorte.particion.value_counts().to_dict())
if LIMITE == 0:
    assert cohorte.particion.value_counts().to_dict() == {"Train": 7123, "Validation": 1262, "Test": 918}
C.check_targets(cohorte)

### 4.2 Variables auxiliares

Hay variables que no entran a la red, pero sirven para ver dónde se equivoca: el tipo morfológico (T_11), el tamaño aparente (D90 = 2 × Petro90) y si la galaxia tiene varias observaciones en Pipe3D.

La última merece una aclaración. Para 114 galaxias con varias observaciones, los archivos de entrenamiento de Katachi usaron una observación distinta a la del catálogo del equipo, y sus etiquetas difieren hasta 0.32 dex en masa y 1.19 dex en SFR. Nosotros usamos las del catálogo del equipo y revisamos ese grupo por separado.

Al final, la muestra se guarda en un CSV. Si el cuaderno se vuelve a ejecutar, se comprueba que las particiones no hayan cambiado.

In [ ]:
from astropy.table import Table
aux = Table.read(REPO_DIR / "Datos" / "inferencia.fits")[["MANGAID", "T_11", "PETRO_TH90"]].to_pandas()
aux["mangaid"] = aux.MANGAID.map(lambda x: x.decode() if isinstance(x, bytes) else x).str.strip()
cohorte = cohorte.merge(aux.drop(columns="MANGAID").drop_duplicates("mangaid"), on="mangaid", how="left")
cohorte["D90_arcsec"] = 2 * cohorte.PETRO_TH90
assert cohorte.ruta_local.map(os.path.exists).all(), "Falta algún FITS"

TAG = f"_lim{LIMITE}" if LIMITE else ""
COH_DIR = guard.mkdir(paths.results / "cohorte")
f_exc, f_coh = COH_DIR / f"exclusiones_qc{TAG}.csv", COH_DIR / f"cohorte{TAG}.csv"
if not f_exc.exists():
    guard.write(f_exc, lambda t: EXCLUIDAS.to_csv(t, index=False))
if not f_coh.exists():
    guard.write(f_coh, lambda t: cohorte.drop(columns=["ruta", "url"], errors="ignore").to_csv(t, index=False))
else:
    previa = pd.read_csv(f_coh, dtype={"mangaid": str}).set_index("mangaid").particion
    assert (previa.reindex(cohorte.mangaid).values == cohorte.particion.values).all(), "Las particiones cambiaron"
print("Con varias observaciones en Pipe3D:", cohorte[cohorte.varias_obs_pipe3d].particion.value_counts().to_dict())

## 5. Las imágenes

### 5.1 Recorte y caché

Cada FITS tiene 800 × 800 píxeles en tres bandas (g, r, z), a 0.262″ por píxel. La red recibe el recorte central de 300 × 300 píxeles (78.6″) a esa misma escala, de modo que ningún píxel se interpola. Leer 74 GB de Drive en cada época sería inviable, así que el recorte de cada galaxia se lee una sola vez y se guarda en una caché. Si se interrumpe, continúa donde quedó.

Aquí entra la *variante*. Con `desi` el cubo no se modifica. Con `desi_prep`, el preprocesamiento del equipo se aplica sobre el cubo completo de 800 píxeles, porque necesita el cielo y los objetos vecinos, y luego se recorta. Cada variante tiene su propia caché. Todo lo demás es idéntico entre variantes, de modo que la diferencia entre E2 y E1 mide solo el efecto del preprocesamiento. Si el preprocesamiento falla en una galaxia, se usa su imagen sin procesar y queda marcada. Si fallan más del 5 %, el cuaderno se detiene.

La caché se guarda en float16. El error que esto introduce en la entrada de la red es como mucho 1e-4, unas 40 veces más fino que la resolución de los PNG de Katachi (1/255). Está verificado en las pruebas.

In [ ]:
fcfg = FitsConfig(crop_px=RECORTE_PX, out_px=SALIDA_PX, beta=BETA)
VAR = get_variante(VARIANTE)
ETIQUETA = "E1" if VAR.is_identity else "E2"
print(f"Variante {VAR.name} · recorte {fcfg.crop_px} px = {fcfg.field_arcsec:.1f}″ · {fcfg.pixscale_out:.3f}″/px")

CACHE_DIR = cache_dir(paths.cache, VAR, fcfg, TAG)
t0 = time.time()
build_cache(cohorte, "ruta_local", CACHE_DIR, guard, fcfg, variant=VAR, chunk=500,
            workers=HILOS if VAR.is_identity else max(1, (os.cpu_count() or 2) - 1))
X_ALL, ids_cache, frac_ceros, respaldo = load_cache(CACHE_DIR, dtype=fcfg.storage_dtype)
assert list(ids_cache) == list(cohorte.mangaid), "El orden de la caché no coincide con la muestra"

cohorte["cobertura_incompleta"] = frac_ceros.max(1) > fcfg.zero_flag_frac
cohorte["respaldo_prep"] = respaldo
print(f"Caché: {X_ALL.shape}, {X_ALL.nbytes / 1e9:.1f} GB, {time.time() - t0:.0f} s")
print("Con más de 5 % de píxeles sin datos:", int(cohorte.cobertura_incompleta.sum()),
      "· preprocesamiento fallido:", int(respaldo.sum()))

### 5.2 Transformación del brillo

El brillo de una galaxia abarca varios órdenes de magnitud, desde un núcleo muy brillante hasta un halo apenas por encima del ruido. Usamos `asinh(flujo / (β·σ_b))`, donde σ_b es el ruido del cielo en la banda b. Cerca de cero esta función es casi lineal, de modo que el ruido y la señal débil no se deforman; para valores grandes se comporta como un logaritmo y comprime el núcleo. Después se divide entre una constante por banda, s_b, para que los valores queden aproximadamente entre 0 y 1, como en las imágenes de Katachi.

Las dos constantes se calculan una sola vez con galaxias de Train y se usan igual en todas las galaxias y en todas las variantes. El flujo cero sigue valiendo cero, igual que el cielo negro de Katachi. Así, cuando la rotación aleatoria rellena las esquinas con cero, eso significa "sin luz" en ambos casos.

In [ ]:
NORM_DIR = guard.mkdir(paths.results / "normalizacion")
f_sigma = NORM_DIR / f"sigma_cielo_salida{fcfg.out_px}{TAG}.json"
if f_sigma.exists():
    SIGMA = np.array(json.loads(f_sigma.read_text())["sigma"])
else:
    tr = cohorte[cohorte.particion == "Train"]
    elegidas = tr.sample(min(fcfg.n_sky_galaxies, len(tr)), random_state=0)
    SIGMA = sky_sigma(list(elegidas.ruta_local), fcfg)            # siempre sobre los cubos sin procesar
    guard.write_json(f_sigma, {"sigma": SIGMA.tolist(), "n": len(elegidas)})

f_norm = NORM_DIR / f"normalizador_recorte{fcfg.crop_px}_beta{BETA:g}{TAG}.json"
if f_norm.exists():
    NORM = Normalizer.from_json(f_norm.read_text())
else:
    if VAR.is_identity:
        X_CRUDO = X_ALL
    else:                                                          # s_b también sale de los cubos sin procesar
        dir_crudo = cache_dir(paths.cache, DESI, fcfg, TAG)
        build_cache(cohorte, "ruta_local", dir_crudo, guard, fcfg, variant=DESI, workers=HILOS)
        X_CRUDO = load_cache(dir_crudo, dtype=fcfg.storage_dtype)[0]
    i_tr = np.where(cohorte.particion == "Train")[0]
    sub = np.sort(np.random.default_rng(0).choice(i_tr, min(1000, len(i_tr)), replace=False))
    NORM = Normalizer(SIGMA, fit_scale(torch.from_numpy(X_CRUDO[sub].astype(np.float32)), fcfg, SIGMA, n_max=len(sub)), BETA)
    guard.write_text(f_norm, NORM.to_json())
print("σ del cielo [nanomaggies/px] (g, r, z):", np.round(NORM.sigma, 5))
print("s_b (g, r, z):", np.round(NORM.scale, 3), "· β =", NORM.beta)

### 5.3 Cómo se ven

Arriba aparece la imagen SDSS que vio Katachi; abajo, la misma galaxia tal como la recibe nuestra red. El campo de abajo es más grande (78.6″ frente a 50″), así que la galaxia se ve más pequeña y aparecen más vecinos. Los histogramas muestran los píxeles antes y después de la transformación: la mayor parte del recorte es cielo, en torno a cero, y la galaxia forma la cola larga hacia la derecha.

In [ ]:
IMGS = load_images_cat(EXT / "images.cat")
ej = np.random.default_rng(1).choice(np.where(cohorte.particion == "Train")[0], 6, replace=False)
entrada = preprocess(torch.from_numpy(X_ALL[ej].astype(np.float32)), NORM, fcfg.out_px).numpy()
R.galeria_sdss_desi(images_tensor(IMGS, cohorte.mangaid.iloc[ej]).numpy(), entrada,
                    [f"{m}\nlog M*={v:.2f}" for m, v in zip(cohorte.mangaid.iloc[ej], cohorte.log_mstar.iloc[ej])],
                    fcfg.field_arcsec, fcfg.out_px)
R.histogramas_entrada(X_ALL[ej[:3]].astype(np.float32), entrada[:3], NORM.sigma)

## 6. Línea base: Katachi publicado (B0)

`scalars.cat` incluye las predicciones de Katachi para cada galaxia. Comprobamos que salen exactamente al pasar sus imágenes, sin giros ni reflejos, por sus pesos publicados; la diferencia es del orden de 1e-4. Por eso también evaluamos nuestra red sin aumentación.

La tabla muestra B0 en tres grupos de galaxias. El primero es el Test completo de Katachi (986). El segundo es ese Test con los cortes del artículo (969), y es el que reproduce el 0.22 dex que reporta el artículo para la masa. El tercero es nuestro Test, que además exige QCFLAG = 0 (918). La comparación final se hace con este último.

In [ ]:
PRED_B0 = {"log_mstar": "pred_mstar", "log_sfr": "pred_sfr", "d4000": "pred_d4000"}
kt = scalars[scalars.split.isin(["Train", "Test"])].copy()
kt["cortes"] = C.katachi_cuts(kt)
grupos_b0 = {"Test de Katachi (986)": kt[(kt.split == "Test") & np.isfinite(kt.d4000)],
             "Test con cortes del artículo (969)": kt[(kt.split == "Test") & kt.cortes],
             "Nuestro Test, QCFLAG = 0 (918)": cohorte[cohorte.particion == "Test"]}
filas = [{"grupo": g, "objetivo": t, **EV.metrics(d[t], d[c])} for g, d in grupos_b0.items() for t, c in PRED_B0.items()]
display(pd.DataFrame(filas)[["grupo", "objetivo", "N", "RMSE", "MAD", "R2", "sesgo", "sigma_NMAD"]])
print("Artículo: RMSE en Test de 0.22 (M*), 0.75 (SFR) y 0.28 (D4000).")

## 7. Entrenamiento

La receta sigue el artículo (§3.1.2):

- Tres ResNet50 iniciadas con los pesos de ImageNet, con las capas preentrenadas congeladas (sección 7.1).
- Cada red tiene su propia pérdida (error cuadrático medio) y su propio Adam, con tasa de aprendizaje 1e-3 y lotes de 32. Las redes de SFR y D4000 reciben la masa (y la SFR) **predichas** por las anteriores, nunca las verdaderas.
- Aumentación: reflejos horizontal y vertical, y una rotación aleatoria de 0 a 360°.
- La tasa de aprendizaje de cada red baja cuando su pérdida de entrenamiento no mejora en 3 épocas.
- Cada red se detiene cuando su pérdida de entrenamiento no mejora más de un 0.1 % en 10 épocas. A partir de ahí queda fija y solo produce su predicción para la siguiente. El entrenamiento termina cuando las tres se han detenido.
- Validation se evalúa en cada época, pero solo para vigilar el sobreajuste.
- Katachi entrenó en dos GPU, así que cada lote de 32 se normalizaba en dos mitades de 16. Lo reproducimos en una sola GPU.

Cada época se guarda un punto de control. Si Colab se desconecta, el entrenamiento sigue en la época siguiente con el mismo estado aleatorio; una prueba verifica que el resultado es idéntico al de una corrida sin interrupciones. Para ir más rápido, los datos viven en la GPU y las tres pérdidas se derivan juntas. Ninguno de esos cambios altera los cálculos, y también están cubiertos por pruebas.

In [ ]:
tcfg = TrainConfig(seed=SEED, max_epochs=MAX_EPOCHS, freeze_pretrained=CONGELAR)
MODO   = f"congelado{tcfg.first_trainable_param}" if CONGELAR else "completo"
RUN_ID = f"{ETIQUETA}_{VAR.key}_recorte{fcfg.crop_px}_beta{BETA:g}_{MODO}_seed{SEED}{TAG}"
RUN_DIR = guard.mkdir(paths.run_dir(RUN_ID))

config_corrida = json.loads(Experiment(paths=paths, fits=fcfg, train=tcfg).to_json())
config_corrida.update(commit=COMMIT, normalizador=json.loads(NORM.to_json()), variante=VAR.manifest())
f_cfg = RUN_DIR / "config.json"
if not f_cfg.exists():
    guard.write_json(f_cfg, config_corrida)
else:   # al reanudar, la configuración debe ser la misma
    previa = json.loads(f_cfg.read_text())
    assert all(previa.get(k) == config_corrida[k] for k in ("train", "fits", "variante")), "La configuración cambió"

Y_ALL = cohorte[list(TARGETS)].to_numpy(np.float32)
idx = {p: np.where(cohorte.particion == p)[0] for p in ["Train", "Validation", "Test"]}
print("Corrida:", RUN_ID, "·", {k: len(v) for k, v in idx.items()})

### 7.1 Congelado de capas

El artículo dice: *"All the layers of this pretrained model remained frozen, and we trained the final convolutional and linear layers of the model"* (§3.1.3). No dice cuántas son las capas finales, ni qué hacer con BatchNorm o con la primera capa de las redes que reciben la masa y la SFR. Cada detalle se resolvió así; el razonamiento completo está en `Documentación/Propuesta_congelado_Katachi.md`.

| | Decisión | Por qué |
|---|---|---|
| Capas entrenables | Desde el tensor 144: las últimas 5 convolucionales (`layer4.1.conv2` en adelante) y la lineal. 7.9 M de 23.5 M parámetros. | Es la única regla concreta que dejaron los autores (`i < 144` en su código). |
| BatchNorm de la parte congelada | En modo evaluación, con las estadísticas de ImageNet | En PyTorch, `requires_grad = False` no impide que la media y la varianza se sigan actualizando; sin esto, las capas "congeladas" cambiarían. Es la práctica recomendada (guía de transferencia de aprendizaje de Keras). |
| Primera capa de SFR y D4000 | Parte de imagen con los pesos de ImageNet, congelada; canales de masa y SFR iniciados en cero y entrenables | Una primera capa aleatoria, como la de Katachi, dejaría sin sentido las capas congeladas que vienen después. Al empezar en cero, la red parte exactamente de la ResNet50 preentrenada. |
| Orden de canales | z, r, g | Los filtros congelados de ImageNet esperan rojo, verde y azul. Katachi les daba R = i, G = r, B = g. |
| Entrada | Sin la normalización de ImageNet | Katachi tampoco la usaba (`ToTensor()`). |

**Una advertencia sobre B0.** Los pesos que publicó Katachi tienen **todas** sus capas entrenadas. Ninguno de sus tensores coincide con ImageNet, y ningún par de sus siete modelos comparte un tensor, cosa imposible si hubieran congelado. Por eso la diferencia E1 − B0 mezcla el cambio de imágenes con el del congelado. Para separarlos, la corrida de control repite todo con `CONGELAR = False`.

La tabla de abajo muestra cuántos parámetros quedan entrenables en cada red. Al terminar el entrenamiento (7.3), se comprueba sobre los pesos guardados que todo lo congelado sigue siendo idéntico, bit a bit, a ImageNet.

In [ ]:
from src.katachi_fits.chain import Chain
if CONGELAR:
    conteo = pd.DataFrame(Chain(imagenet=True, ghost_splits=1, freeze_first=tcfg.first_trainable_param).freeze_report).T
    conteo["% entrenable"] = (100 * conteo.entrenables / (conteo.entrenables + conteo.congelados)).round(1)
    display(conteo)
else:
    print("Corrida de control: todas las capas se entrenan.")

### 7.2 Entrenamiento

Antes de lanzar el entrenamiento, una medición rápida de cuánto tarda una época en esta GPU, con un modelo que se descarta enseguida. Con ese dato se estima la duración total.

In [ ]:
if ENTRENAR and DEVICE.type == "cuda" and not (RUN_DIR / "final_model.pt").exists():
    prueba = Trainer(TrainConfig(seed=999), NORM, fcfg.out_px, RUN_DIR / "_prueba", guard, DEVICE, log=lambda *_: None)
    n = min(32 * 40, len(idx["Train"]))
    datos = GPUData(X_ALL[idx["Train"][:n]], Y_ALL[idx["Train"][:n]], DEVICE)
    prueba.train_epoch(GPUData(datos.x[:64], datos.y[:64], DEVICE))      # calentamiento
    torch.cuda.synchronize(); t0 = time.time(); prueba.train_epoch(datos); torch.cuda.synchronize()
    seg = (time.time() - t0) / n * len(idx["Train"])
    print(f"≈ {seg / 60:.1f} min por época · a 170 épocas, ≈ {seg * 170 / 3600:.1f} h")
    del prueba, datos; torch.cuda.empty_cache()

In [ ]:
trainer = Trainer(tcfg, NORM, fcfg.out_px, RUN_DIR, guard, DEVICE, log=lambda s: print(s, flush=True))
if ENTRENAR and not (RUN_DIR / "final_model.pt").exists():
    TR = GPUData(X_ALL[idx["Train"]], Y_ALL[idx["Train"]], DEVICE)
    VA = GPUData(X_ALL[idx["Validation"]], Y_ALL[idx["Validation"]], DEVICE) if len(idx["Validation"]) else None
    trainer.fit(TR, VA)
    del TR, VA
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
trainer.load_final()
historia = pd.read_csv(RUN_DIR / "history.csv")
print("Épocas entrenadas:", len(historia))

**Cómo leer las curvas.** Arriba, la pérdida de cada red: en Train (con aumentación) es la que guía la receta; en Validation (sin aumentación) solo sirve para vigilar. Si Train sigue bajando mientras Validation sube, la red está memorizando el conjunto de entrenamiento. Abajo, la tasa de aprendizaje de cada red: cada escalón es una reducción. La línea punteada marca la época en que cada red se detuvo.

In [ ]:
R.curvas_aprendizaje(historia)

### 7.3 Comprobación del congelado

Es la misma prueba que reveló que los pesos de Katachi no estaban congelados, aplicada ahora a los nuestros. Para cada red, todos los tensores congelados (los 144 parámetros y los buffers de sus 48 BatchNorm) deben ser idénticos bit a bit a los de ImageNet, y todos los entrenables deben haber cambiado. Si algo no cumple, el cuaderno se detiene.

In [ ]:
if CONGELAR:
    import torchvision
    imagenet = torchvision.models.resnet50(weights="IMAGENET1K_V1").state_dict()
    auditoria = freezing_audit(trainer.model.cpu(), tcfg.first_trainable_param, imagenet)
    trainer.model.to(DEVICE)
    display(auditoria)
    for _, r in auditoria.iterrows():
        ok, total = map(int, r["congelados idénticos a ImageNet"].split("/"))
        assert ok == total, f"La red {r.red} tiene capas congeladas que cambiaron"
else:
    print("Corrida de control: no hay capas congeladas.")

Con el modelo final calculamos las predicciones, siempre sin aumentación. Test solo se predice si `ABRIR_TEST = True`. Las predicciones se guardan junto con las de Katachi para poder volver a analizarlas sin reentrenar.

In [ ]:
partes = ["Train", "Validation"] + (["Test"] if ABRIR_TEST else [])
PRED_E1 = {"log_mstar": "e1_mstar", "log_sfr": "e1_sfr", "d4000": "e1_d4000"}
for c in PRED_E1.values():
    cohorte[c] = np.nan
for p in partes:
    if len(idx[p]):
        cohorte.loc[idx[p], list(PRED_E1.values())] = trainer.predict(GPUData(X_ALL[idx[p]], Y_ALL[idx[p]], DEVICE))

f_pred = RUN_DIR / f"predicciones_{'con' if ABRIR_TEST else 'sin'}_test.csv"
if not f_pred.exists():
    cols = ["mangaid", "particion", "respaldo_prep", *TARGETS, *PRED_E1.values(), *PRED_B0.values()]
    guard.write(f_pred, lambda t: cohorte[cols].to_csv(t, index=False))
print("Predicciones guardadas en", f_pred.name)

## 8. Resultados

Mientras Test siga cerrado, esta sección muestra solo Validation y solo para nuestra red. Katachi no se compara en Validation porque entrenó con esas galaxias.

Cuando se activa `ABRIR_TEST`, la primera ejecución deja escrito en `apertura_test.json` cuándo y con qué commit se abrió Test. Ese archivo no se vuelve a escribir.

In [ ]:
if ABRIR_TEST:
    f_abierto = RUN_DIR / "apertura_test.json"
    if not f_abierto.exists():
        guard.write_json(f_abierto, {"fecha_utc": datetime.datetime.utcnow().isoformat(), "commit": COMMIT, "corrida": RUN_ID})
    print("Test abierto:", json.loads(f_abierto.read_text()))

PARTE = "Test" if ABRIR_TEST else "Validation"
EV_DF = cohorte[cohorte.particion == PARTE].copy()
MODELOS = {f"{ETIQUETA} (DESI)": PRED_E1} | ({"B0 (Katachi)": PRED_B0} if ABRIR_TEST else {})
TABLA = R.tabla_metricas(EV_DF, MODELOS, N_BOOT)
print("Partición evaluada:", PARTE)
display(TABLA)

### 8.1 Predicción contra valor verdadero

Es la figura 3 del artículo de Katachi. Cada panel muestra la estimación contra el valor de Pipe3D; la línea roja es la predicción perfecta. Si la nube se aplana respecto a esa línea (pendiente menor que 1), la red está empujando sus estimaciones hacia el promedio. Es un efecto típico cuando la imagen no contiene suficiente información para estimar la propiedad.

In [ ]:
R.figura3({"Train": cohorte[cohorte.particion == "Train"], PARTE: EV_DF}, PRED_E1,
          f"{ETIQUETA}: cadena de Katachi entrenada con FITS de DESI")
if ABRIR_TEST:
    R.figura3({"Train (Katachi)": kt[(kt.split == "Train") & kt.cortes], PARTE: EV_DF}, PRED_B0,
              "B0: Katachi publicado (SDSS)")

### 8.2 Comparación con Katachi

Aquí se aplica la regla de decisión del §1. Ambas redes se evalúan sobre exactamente las mismas galaxias, así que la diferencia se calcula galaxia por galaxia. Esto elimina la variación debida a qué galaxias cayeron en Test. El intervalo sale de remuestrear esas galaxias 10,000 veces. La prueba de Wilcoxon y la corrección de Holm (por evaluar tres propiedades) se dan como apoyo.

El artículo también reporta la SFR solo para galaxias con formación estelar activa (sSFR > −10.8). No dice si usó la sSFR verdadera o la estimada; aquí usamos la verdadera.

In [ ]:
if ABRIR_TEST:
    display(R.comparacion_pareada(EV_DF, PRED_E1, PRED_B0, N_BOOT))
    activas = EV_DF[(EV_DF.log_sfr - EV_DF.log_mstar) > K.SSFR_CUT]
    print(f"SFR con sSFR > {K.SSFR_CUT} (N={len(activas)}): "
          f"RMSE {ETIQUETA} = {EV.metrics(activas.log_sfr, activas.e1_sfr)['RMSE']:.3f}, "
          f"B0 = {EV.metrics(activas.log_sfr, activas.pred_sfr)['RMSE']:.3f} (artículo: 0.31)")
else:
    print("Test cerrado.")

### 8.3 Dónde se equivoca

Un único número puede esconder errores que se concentran en un tipo de galaxia. Por eso calculamos el sesgo y la dispersión en intervalos de masa, de redshift, de tipo morfológico y de tamaño aparente. Este último compara el diámetro D90 con el campo, que es de 78.6″ para nuestra red y de 50″ para Katachi. También separamos tres grupos: las galaxias con varias observaciones en Pipe3D (donde Katachi entrenó con otras etiquetas), las que tienen zonas sin datos y las de cada grupo de calidad.

In [ ]:
R.residuos_por_intervalos(EV_DF, MODELOS, {
    "log_mstar":  [8, 9.5, 10, 10.5, 11, 12],
    "redshift":   [0, 0.025, 0.035, 0.05, 0.08, 0.2],
    "T_11":       [-10, 0, 2, 4, 6, 12],
    "D90_arcsec": [0, 25, 50, 78.6, 400],
})
for grupo in ["varias_obs_pipe3d", "cobertura_incompleta"]:
    display(R.rmse_por_grupo(EV_DF, grupo, MODELOS))

### 8.4 El límite que imponen las etiquetas

Ninguna red puede ser más precisa que sus etiquetas. Pipe3D reporta un error para cada valor; si la dispersión de una red se acerca a ese error, queda poco margen para mejorar con mejores imágenes.

In [ ]:
for t, e in [("log_mstar", "error_log_mstar"), ("log_sfr", "error_log_sfr"), ("d4000", "error_d4000")]:
    linea = f"{t:10s} error mediano de Pipe3D = {EV_DF[e].median():.3f}"
    for nombre, cols in MODELOS.items():
        linea += f" · σ_NMAD {nombre} = {EV.metrics(EV_DF[t], EV_DF[cols[t]])['sigma_NMAD']:.3f}"
    print(linea)

### 8.5 Con preprocesamiento (E2) frente a sin él (E1)

Esta comparación solo se ejecuta cuando la variante es distinta de `desi`. Se busca la corrida E1 con la misma semilla, recorte y β, y se comparan las dos sobre las mismas galaxias con la misma regla de decisión. También se muestra el resultado sin las galaxias donde falló el preprocesamiento.

In [ ]:
if not VAR.is_identity:
    run_e1 = paths.run_dir(RUN_ID.replace("E2_", "E1_", 1).replace(VAR.key, DESI.key, 1))
    f_e1 = run_e1 / f"predicciones_{'con' if ABRIR_TEST else 'sin'}_test.csv"
    assert f_e1.exists(), f"Falta la corrida E1 equivalente: {run_e1.name}"
    e1 = pd.read_csv(f_e1, dtype={"mangaid": str}).set_index("mangaid")[list(PRED_E1.values())].add_suffix("_E1")
    D2 = EV_DF.join(e1, on="mangaid")
    ref = {t: c + "_E1" for t, c in PRED_E1.items()}
    for nombre, d in [("todas las galaxias", D2), ("sin preprocesamiento fallido", D2[~D2.respaldo_prep])]:
        print(f"E2 frente a E1 · {nombre}")
        display(R.comparacion_pareada(d, PRED_E1, ref, N_BOOT))
else:
    print("Esta es la corrida E1; la comparación se hará cuando exista una corrida E2.")

## 9. Edad y historia de formación estelar

Katachi convierte D4000 en t50, el momento en que la galaxia había formado la mitad de su masa, con una pequeña red que recibe D4000 y la SFR específica (SFR/M\*). Después reconstruye la historia de formación estelar con `dense_basis`. Usamos su red publicada tal cual, porque no depende de las imágenes.

Al revisar su código encontramos algo importante: las historias publicadas por Katachi se calcularon con los valores verdaderos de Pipe3D, no con lo que predice a partir de las imágenes. Por eso calculamos tres versiones:

1. **(a) etiquetas:** lo que hizo el código de Katachi; es la referencia.
2. **(b) B0:** lo que describe su artículo, a partir de sus predicciones.
3. **(c) nuestra red.**

La primera línea de la salida confirma que la versión (a) reproduce el t50 publicado.

In [ ]:
T50 = load_t50(EXT / "d400_to_t50.pytorch")
D = EV_DF.copy()
D["t50_a"] = predict_t50(T50, D.log_mstar, D.log_sfr, D.d4000)
D["t50_c"] = predict_t50(T50, D.e1_mstar, D.e1_sfr, D.e1_d4000)
VERSIONES = [("t50_a", "log_mstar", "log_sfr", "(a) etiquetas")]
if ABRIR_TEST:
    D["t50_b"] = predict_t50(T50, D.pred_mstar, D.pred_sfr, D.pred_d4000)
    VERSIONES.append(("t50_b", "pred_mstar", "pred_sfr", "(b) B0"))
VERSIONES.append(("t50_c", "e1_mstar", "e1_sfr", f"(c) {ETIQUETA}"))

print("(a) frente al t50 publicado: diferencia máxima", f"{np.nanmax(np.abs(D.t50_a - D.t50_model)):.1e} Gyr")
for col, _, _, nombre in VERSIONES[1:]:
    m = EV.metrics(D.t50_a, D[col])
    print(f"t50 {nombre} frente a (a): RMSE {m['RMSE']:.2f} Gyr · sesgo {m['sesgo']:+.2f} Gyr")
R.figura4_t50(D, VERSIONES)

Con t50, la masa y la SFR de cada versión, reconstruimos dónde estaba cada galaxia en el plano masa–SFR hace 0.1, 1 y 10 mil millones de años. Ese resultado alimenta la figura 8 de la sección 11.

In [ ]:
LOOKBACK = [0.1, 1.0, 10.0]
PLANOS = {nombre: R.plano_en_el_pasado(D, mx, sx, col, LOOKBACK) for col, mx, sx, nombre in VERSIONES}
print("Historias reconstruidas:", {k: len(v) for k, v in PLANOS.items()})

## 10. Robustez

Repetimos las dos pruebas de la sección 4.1.2 de Katachi.

**Ruido.** A cada imagen se le suma ruido gaussiano con la desviación estándar de su borde de 5 píxeles, 100 veces, y se mide cuánto varía cada estimación. Katachi reporta 0.023 (masa), 0.096 (SFR), 0.012 (D4000) y 0.021 (t50, en escala logarítmica). Como prueba adicional, repetimos el ejercicio con el ruido real del cielo de cada banda, medido en el FITS.

**Resolución.** Reducimos la resolución por 2, 4 y 8 y la devolvemos a su tamaño original. Katachi encontró poco efecto hasta un factor de 4.

In [ ]:
T50_GPU = T50.to(DEVICE)
crudo_ev = torch.from_numpy(X_ALL[EV_DF.index.to_numpy()]).to(DEVICE)
with torch.no_grad():
    X_EV = torch.cat([network_input(crudo_ev[i:i + 128].float(), NORM, fcfg.out_px, tcfg.input_band_order)
                      for i in range(0, len(crudo_ev), 128)])                       # orden de la red (z, r, g)
E1_RED = trainer.model.eval()
REDES = {ETIQUETA: (R.cadena_con_t50(E1_RED, T50_GPU, K.T50_SCALE_GYR), X_EV)}
if ABRIR_TEST:
    B0_RED = katachi_chain(EXT / "Mass_ResNet50_log_t50_chain_90_10.pytorch", EXT / "SFR_ResNet50_log_t50_chain_90_10.pytorch",
                           EXT / "d4000_d4000_chain.pytorch").to(DEVICE)
    X_B0 = images_tensor(IMGS, EV_DF.mangaid).to(DEVICE)
    REDES["B0"] = (R.cadena_con_t50(B0_RED, T50_GPU, K.T50_SCALE_GYR), X_B0)

columnas = ["M*", "SFR", "D4000", "t50 [Gyr]", "log t50"]
filas = [{"modelo": n, "ruido": "borde de 5 px (Katachi)", **dict(zip(columnas, RB.noise_bootstrap(f, x, n_iter=N_RUIDO).mean(0)))}
         for n, (f, x) in REDES.items()]
s_b = torch.as_tensor(NORM.scale.astype(np.float32), device=DEVICE).view(1, -1, 1, 1)
sigma_b = torch.as_tensor(NORM.sigma.astype(np.float32), device=DEVICE).view(1, -1, 1, 1)
orden = band_permutation(tcfg.input_band_order)
en_flujo = lambda xb: REDES[ETIQUETA][0]((stretch(xb, NORM.sigma, NORM.beta) / s_b)[:, orden])
sd = RB.noise_bootstrap(en_flujo, resample(crudo_ev.float(), fcfg.out_px), n_iter=N_RUIDO, sigma=sigma_b)
filas.append({"modelo": ETIQUETA, "ruido": "ruido real del cielo", **dict(zip(columnas, sd.mean(0)))})
filas.append({"modelo": "Katachi (artículo)", "ruido": "borde de 5 px", "M*": 0.023, "SFR": 0.096, "D4000": 0.012, "log t50": 0.021})
display(pd.DataFrame(filas))

In [ ]:
R.figura_degradacion(R.degradacion(REDES, EV_DF))

## 11. ¿En qué partes de la imagen se apoya la red?

Un mapa SHAP asigna a cada píxel cuánto empuja la estimación hacia arriba (rojo) o hacia abajo (azul). Seguimos el procedimiento de Katachi, que reconstruimos a partir de su código. Se usa `GradientExplainer` con 32 imágenes de referencia. El mapa se suaviza y se promedia en anillos alrededor del centro para obtener un perfil radial. De ese perfil se define el gradiente ∇ como su valor a 20 kpc menos su valor en el centro: si es negativo, la red se apoya sobre todo en el bulbo; si es positivo, en las regiones externas.

Hay dos detalles que conviene aclarar.

1. **La escala en kpc de Katachi está mal.** Convierte píxeles a kpc suponiendo 0.5″ por píxel (el tamaño de un spaxel de MaNGA), pero sus imágenes tienen 0.195″ por píxel. Su eje radial queda estirado 2.56 veces, y su "20 kpc" equivale en realidad a unos 8 kpc. Para comparar nuestra red con la suya usamos la escala real de cada una; la convención de Katachi solo se usa para reproducir sus figuras. Si los 20 kpc caen fuera de la imagen, esa galaxia queda sin gradiente.
2. **Qué red se explica.** Katachi explicó redes de SFR y D4000 que solo ven la imagen. Nosotros no entrenamos esas redes, así que explicamos la cadena completa, que incluye el efecto de cada píxel a través de la masa y la SFR estimadas. Para B0 calculamos las dos versiones; la de cadena es la que se compara con la nuestra.

In [ ]:
rng = np.random.default_rng(0)
n_fondo = min(K.SHAP_BACKGROUND, len(EV_DF) // 2)
n_explicar = min(N_SHAP, len(EV_DF) - n_fondo)
orden = rng.permutation(len(EV_DF))
I_FONDO, I_SHAP = orden[:n_fondo], np.sort(orden[n_fondo:n_fondo + n_explicar])
SH = EV_DF.iloc[I_SHAP].copy()
print(f"Se explican {n_explicar} galaxias; otras {n_fondo} sirven de referencia.")

nombres = R.NOMBRES_RED
CLAVE = f"{ETIQUETA} (cadena)"
SHAP = {CLAVE: R.mapas_shap({n: ChainTarget(E1_RED, k) for k, n in enumerate(nombres)}, X_EV, I_FONDO, I_SHAP, N_SHAP_MUESTRAS)}
ESCALA = {CLAVE: fcfg.pixscale_out}
if ABRIR_TEST:
    SHAP["B0 (cadena)"] = R.mapas_shap({n: ChainTarget(B0_RED, k) for k, n in enumerate(nombres)}, X_B0, I_FONDO, I_SHAP, N_SHAP_MUESTRAS)
    solo = {"M*": B0_RED.mass, "SFR": katachi_solo(EXT / "sfr_solo.pytorch").to(DEVICE),
            "D4000": katachi_solo(EXT / "d4000.pytorch").to(DEVICE)}
    SHAP["B0 (como Katachi)"] = R.mapas_shap(solo, X_B0, I_FONDO, I_SHAP, N_SHAP_MUESTRAS)
    ESCALA |= {"B0 (cadena)": K.KATACHI_PIXSCALE_ARCSEC, "B0 (como Katachi)": K.KATACHI_PIXSCALE_ARCSEC}

GR = {k: R.gradientes(v, SH.redshift.to_numpy(), ESCALA[k],
                      K.KATACHI_KPC_ARCSEC_PER_PX if k.startswith("B0") else None, SH.index)
      for k, v in SHAP.items()}
for k, g in GR.items():
    print(f"{k}: {int(np.isfinite(g['grad_M*_real']).sum())} de {len(g)} galaxias con gradiente a 20 kpc")

**Figura 5.** La imagen de cinco galaxias y sus mapas para cada propiedad.

In [ ]:
with torch.no_grad():   # para mostrar se usa el orden g, r, z
    X_VIS = preprocess(crudo_ev[torch.as_tensor(I_SHAP, device=crudo_ev.device)].float(), NORM, fcfg.out_px).cpu().numpy()
R.figura5_mapas(X_VIS, SH, SHAP)

**Figura 6.** El perfil radial mediano de los mapas, separado en galaxias de masa baja, media y alta.

In [ ]:
R.figura6_perfiles(SH, GR, ESCALA)

**Figura 7.** El gradiente de cada galaxia sobre el plano masa–SFR. En Katachi, las galaxias por encima y por debajo de unos 10^10.5 M⊙ muestran gradientes opuestos para la masa y la SFR. Debajo de la figura se mide si nuestra red y la de Katachi asignan gradientes parecidos a cada galaxia.

In [ ]:
paneles = [(k, g, "real") for k, g in GR.items()]
if ABRIR_TEST:
    paneles.append(("B0 · convención de Katachi", GR["B0 (como Katachi)"], "katachi"))
R.figura7_gradientes(SH, paneles)
if ABRIR_TEST:
    display(R.comparar_gradientes(GR[CLAVE], GR["B0 (cadena)"]))

**Figura 8.** Dónde estaban las galaxias en el plano masa–SFR en el pasado, según cada versión de la historia (sección 9), coloreadas por el gradiente de masa de nuestra red. Katachi observa que las galaxias con morfología parecida siguen agrupadas en el pasado.

In [ ]:
pos = [EV_DF.index.get_loc(i) for i in SH.index]
R.figura8_pasado(PLANOS, pos, np.asarray(GR[CLAVE]["grad_M*_real"], float), LOOKBACK, f"∇M* ({CLAVE})")

**Por banda y controles.** El mapa de masa separado por canal de entrada corresponde a la sección 4.2.3 de Katachi (nuestra red recibe z, r, g; la de Katachi, i, r, g). Después vienen sus dos controles. Las imágenes de puro ruido deberían dar mapas mucho más débiles que las reales. Y si la imagen se rota 90°, el mapa debería rotar igual, lo que se traduce en una correlación cercana a 1.

In [ ]:
R.figura_shap_por_banda(SHAP, ESCALA, tcfg.input_band_order)
print(R.controles_shap(ChainTarget(E1_RED, 0), X_EV[I_FONDO], X_EV[I_SHAP[:8]], N_SHAP_MUESTRAS))

## 12. Discusión

*Esta sección se escribe después de abrir Test y cita solo números que aparecen arriba.* Para cada punto, el lugar de donde sale:

1. **Resultado principal** (8.2): la diferencia de RMSE en masa con su intervalo y su veredicto; lo mismo para SFR y D4000.
2. **Dónde cambia el error** (8.3): por masa, redshift, morfología, tamaño y los grupos especiales.
3. **Distancia al límite de las etiquetas** (8.4).
4. **Edad e historia** (9): cuánto error hereda t50 de cada red. Recordar que las historias publicadas por Katachi usan etiquetas, no predicciones.
5. **Robustez** (10), frente a los valores del artículo.
6. **Mapas SHAP** (11): si las dos redes se apoyan en las mismas regiones, con la advertencia sobre la escala en kpc de Katachi.
7. **Limitaciones.**
   - Una sola semilla.
   - Katachi no se reentrenó con SDSS, así que la diferencia mezcla el cambio de imágenes con el de procedimiento. En particular, nuestra red está congelada como dice el artículo y los pesos publicados no lo están; la corrida de control (`CONGELAR = False`) separa ese efecto.
   - Un campo de 78.6″ en lugar de 50″.
   - El filtro QCFLAG = 0, que Katachi no usaba.
   - El ajuste de la tasa de aprendizaje de Katachi vio su Test, lo que lo favorece ligeramente.
   - Cuatro galaxias quedaron fuera de la descarga sin explicación conocida.
   - Sin corrección por extinción.
   - Etiquetas de Pipe3D con sus propios errores.

## 13. Reproducibilidad

Se comprueba que las entradas no cambiaron durante la ejecución y se guarda un resumen con la corrida, el commit, la GPU y las métricas. Debajo se listan los archivos de la corrida.

In [ ]:
assert sha256(PLAN_CSV) == SHA_PLAN and sha256(CATALOGO) == SHA_CAT, "Una entrada cambió durante la ejecución"
resumen = {"corrida": RUN_ID, "commit": COMMIT, "test_abierto": ABRIR_TEST, "epocas": int(len(historia)),
           "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
           "metricas": TABLA.drop(columns=["RMSE_IC95"]).to_dict("records")}
guard.write_json(RUN_DIR / f"resumen_{'con' if ABRIR_TEST else 'sin'}_test_{datetime.datetime.utcnow():%Y%m%dT%H%M%S}.json", resumen)
print("Entradas sin cambios. Archivos de la corrida:")
for p in sorted(RUN_DIR.iterdir()):
    print(f"  {p.name:50s} {p.stat().st_size / 1e6:8.1f} MB")